# 00 – YOLOX smoke test (Colab, GPU)

Cél: a tanítás előtt ellenőrizni, hogy a `pixeltable-yolox==0.4.2` működik-e a Colab aktuális
környezetével (Python 3.13, torch 2.11). Kb. 10–15 perc.

**Lépések:** telepítés → Roboflow-letöltés (Safety Food, kis részhalmaz) → átalakítás YOLOX-szerkezetre →
COCO-előtanított modell próbája → 1 epoch finomhangolás kiértékeléssel → folytatás checkpointból →
checkpoint betöltése → ONNX-export próba → `smoke_result.json`.

**Előfeltételek:**
- Futtatókörnyezet: *Runtime → Change runtime type → GPU* (T4 elég).
- Colab Secrets: `ROBOFLOW_API_KEY` (bal oldali kulcs ikon, *Notebook access* bekapcsolva).
- A repó fent van GitHubon; állítsd be a `REPO_URL`-t az első kódcellában.

Futtatás: *Runtime → Run all*. Minden lépés hibatűrő: ha egy elszáll, a többi lefut, ami tud,
és a végén a `smoke_result.json` megmutatja, mi ment és mi nem.
A végén a kiírt JSON-t másold be a repo `reports/smoke_result.json` fájljába.

In [ ]:
# --- Settings ---------------------------------------------------------------
REPO_URL = "https://github.com/CHANGE-ME/ppe-demo.git"
REPO_BRANCH = "phase-0-skeleton"
SMOKE_SOURCE = "safety_food"  # a source name from configs/datasets.yaml (head model)
N_TRAIN, N_VAL = 200, 50
BATCH_SIZE = 16
SEED = 42

DRIVE_SMOKE = "/content/drive/MyDrive/ppe-demo/smoke"
REPO_DIR = "/content/ppe-demo"
RF_DIR = "/content/rf_smoke"  # raw Roboflow download (local disk)
DATA_DIR = "/content/smoke_coco"  # converted dataset (local disk)
OUTPUT_DIR = f"{DRIVE_SMOKE}/out"  # YOLOX checkpoints and logs (Drive)
RESULT_PATH = f"{DRIVE_SMOKE}/smoke_result.json"

In [ ]:
# --- Drive + result recorder ---------------------------------------------------
import contextlib
import datetime as dt
import json
import os
import sys
import time
import traceback
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_SMOKE).mkdir(parents=True, exist_ok=True)

RESULT = {
    "created": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
    "env": {},
    "dataset": {},
    "steps": {},
    "metrics": {},
}


def save_result() -> None:
    Path(RESULT_PATH).write_text(json.dumps(RESULT, indent=2, ensure_ascii=False))


@contextlib.contextmanager
def step(name: str):
    """Run a block, record ok/fail and duration, never re-raise."""
    print(f"\n=== {name} ===")
    t0 = time.time()
    entry = {"status": "ok"}
    try:
        yield entry
    except Exception as exc:  # noqa: BLE001 - smoke test records every failure
        entry["status"] = "fail"
        entry["error"] = f"{type(exc).__name__}: {exc}"[:2000]
        entry["traceback_tail"] = traceback.format_exc()[-3000:]
        print(f"FAILED: {entry['error']}")
    finally:
        entry["seconds"] = round(time.time() - t0, 1)
        RESULT["steps"][name] = entry
        save_result()
        print(f"--> {entry['status']} ({entry['seconds']} s)")


def ok(name: str) -> bool:
    return RESULT["steps"].get(name, {}).get("status") == "ok"

In [ ]:
# --- 1. Clone repo and install `.[train]` (includes pixeltable-yolox) ---------
import subprocess


def run(cmd: list[str], **kwargs) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if proc.returncode != 0:
        raise RuntimeError(
            f"{' '.join(cmd[:4])}... exited {proc.returncode}\n"
            f"{(proc.stdout + proc.stderr)[-2500:]}"
        )
    return proc


with step("install") as s:
    if Path(REPO_DIR, ".git").exists():
        run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
        run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"])
    else:
        run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR])
    s["commit"] = run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).stdout.strip()
    run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[train]"])
    if REPO_DIR + "/src" not in sys.path:
        sys.path.insert(0, REPO_DIR + "/src")  # editable install is not on sys.path until restart

In [ ]:
# --- 2. Environment versions -------------------------------------------------------
from importlib.metadata import PackageNotFoundError, version


def pkg_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


with step("environment") as s:
    import torch

    RESULT["env"] = {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        **{
            p: pkg_version(p)
            for p in [
                "torchvision",
                "pixeltable-yolox",
                "pycocotools",
                "numpy",
                "opencv-python",
                "opencv-python-headless",
                "onnx",
                "onnxsim",
                "roboflow",
            ]
        },
    }
    print(json.dumps(RESULT["env"], indent=2))
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU: Runtime -> Change runtime type -> GPU")

In [ ]:
# --- 3. Roboflow download (COCO format) -----------------------------------------------
# The API key comes from Colab Secrets and is never printed.
with step("roboflow_download") as s:
    from google.colab import userdata
    from roboflow import Roboflow

    from ppe.config import load_datasets

    ds_cfg = load_datasets(Path(REPO_DIR, "configs", "datasets.yaml"))
    HEAD_CLASSES = ds_cfg.models.head.classes
    SOURCE = next(src for src in ds_cfg.models.head.sources if src.name == SMOKE_SOURCE)

    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    version_obj = rf.workspace(SOURCE.workspace).project(SOURCE.project).version(SOURCE.version)
    version_obj.download("coco", location=RF_DIR, overwrite=False)

    ann_files = sorted(Path(RF_DIR).rglob("_annotations.coco.json"))
    s["annotation_files"] = [str(p.relative_to(RF_DIR)) for p in ann_files]
    print(s["annotation_files"])
    if not ann_files:
        raise RuntimeError(f"no _annotations.coco.json under {RF_DIR}")

In [ ]:
# --- 4. Convert to the pixeltable-yolox layout -----------------------------------------
# DATA_DIR/annotations/instances_{train,val}2017.json + DATA_DIR/{train,val}2017/
# Categories are written in configs/datasets.yaml order with ids 1..N, so class index i
# (sorted category ids) == HEAD_CLASSES[i]. Quick conversion only: no group split, no
# pseudo-labelled persons (that is phase 1).
import random
import shutil
from collections import Counter, defaultdict


def convert_split(src_json: Path, src_names: list[str], dst_name: str, n_max: int, rng) -> dict:
    coco = json.loads(src_json.read_text())
    src_cats = {c["id"]: c["name"] for c in coco["categories"]}
    dst_ids = {name: i + 1 for i, name in enumerate(HEAD_CLASSES)}

    anns_by_img = defaultdict(list)
    unmapped = Counter()
    for a in coco["annotations"]:
        src_name = src_cats.get(a["category_id"])
        target = SOURCE.class_map.get(src_name)
        if target is None:
            unmapped[src_name] += 1
            continue
        x, y, w, h = a["bbox"]
        if w <= 1 or h <= 1:
            continue
        anns_by_img[a["image_id"]].append((dst_ids[target], [x, y, w, h]))

    images = [
        im for im in coco["images"] if im["id"] in anns_by_img and im["file_name"] in src_names
    ]
    rng.shuffle(images)
    images = images[:n_max]

    img_dir = Path(DATA_DIR, f"{dst_name}2017")
    img_dir.mkdir(parents=True)
    out_images, out_anns, counts = [], [], Counter()
    for new_id, im in enumerate(images, start=1):
        shutil.copy2(src_json.parent / im["file_name"], img_dir / im["file_name"])
        out_images.append(
            {
                "id": new_id,
                "file_name": im["file_name"],
                "width": im["width"],
                "height": im["height"],
            }
        )
        for cat_id, bbox in anns_by_img[im["id"]]:
            out_anns.append(
                {
                    "id": len(out_anns) + 1,
                    "image_id": new_id,
                    "category_id": cat_id,
                    "bbox": bbox,
                    "area": bbox[2] * bbox[3],
                    "iscrowd": 0,
                }
            )
            counts[HEAD_CLASSES[cat_id - 1]] += 1

    out = {
        "info": {"description": f"ppe-demo smoke test ({SMOKE_SOURCE})"},
        "licenses": [],
        "images": out_images,
        "annotations": out_anns,
        "categories": [{"id": i, "name": n, "supercategory": "none"} for n, i in dst_ids.items()],
    }
    ann_dir = Path(DATA_DIR, "annotations")
    ann_dir.mkdir(parents=True, exist_ok=True)
    (ann_dir / f"instances_{dst_name}2017.json").write_text(json.dumps(out))
    return {
        "images": len(out_images),
        "boxes": dict(counts),
        "unmapped_source_classes": dict(unmapped),
    }


with step("convert") as s:
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    rng = random.Random(SEED)
    by_split = {p.parent.name: p for p in ann_files}
    train_json = by_split.get("train", ann_files[0])
    val_json = by_split.get("valid") or by_split.get("val") or by_split.get("test")

    if val_json is None:  # only one split in the export: carve val out of train
        names = [im["file_name"] for im in json.loads(train_json.read_text())["images"]]
        rng.shuffle(names)
        train_names, val_names = names[N_VAL:], names[:N_VAL]
        val_json = train_json
    else:
        train_names = [im["file_name"] for im in json.loads(train_json.read_text())["images"]]
        val_names = [im["file_name"] for im in json.loads(val_json.read_text())["images"]]

    RESULT["dataset"] = {
        "source": SMOKE_SOURCE,
        "roboflow_version": SOURCE.version,
        "classes": HEAD_CLASSES,
        "train": convert_split(train_json, train_names, "train", N_TRAIN, rng),
        "val": convert_split(val_json, val_names, "val", N_VAL, rng),
    }
    print(json.dumps(RESULT["dataset"], indent=2))
    if RESULT["dataset"]["train"]["images"] == 0:
        raise RuntimeError("no training images after class mapping")

In [ ]:
# --- 5. COCO-pretrained YOLOX-s: download and inference --------------------------------
with step("pretrained_inference") as s:
    from yolox.models import Yolox

    model = Yolox.from_pretrained("yolox_s")  # downloads to ~/.cache/yolox/weights/yolox_s.pth
    PRETRAINED_PTH = Path.home() / ".cache" / "yolox" / "weights" / "yolox_s.pth"
    s["weights_file_exists"] = PRETRAINED_PTH.exists()

    sample = next(Path(DATA_DIR, "train2017").iterdir())
    out = model([str(sample)])
    s["detections_on_sample"] = len(out[0]["labels"])
    s["coco_person_detections"] = sum(1 for lbl in out[0]["labels"] if int(lbl) == 0)
    print(s)
    del model

In [ ]:
# --- 6. Fine-tune 1 epoch (with evaluation) ----------------------------------------------
# Config class: exps/smoke_yolox_s.py (-c module:Class). Eval runs after the epoch, which
# also checks the pycocotools 'info' KeyError (pixeltable-yolox issue #14).
import re

RUN_DIR = Path(OUTPUT_DIR, "smoke_yolox_s")
TRAIN_ENV = {
    **os.environ,
    "PYTHONPATH": f"{REPO_DIR}/exps",
    "PPE_SMOKE_DATA_DIR": DATA_DIR,
    "PPE_SMOKE_OUTPUT_DIR": OUTPUT_DIR,
}


def yolox_train(max_epoch: int, extra: list[str], log_name: str) -> None:
    env = {**TRAIN_ENV, "PPE_SMOKE_MAX_EPOCH": str(max_epoch)}
    cmd = [
        "yolox",
        "train",
        "-c",
        "smoke_yolox_s:SmokeYoloxS",
        "-d",
        "1",
        "-b",
        str(BATCH_SIZE),
        "--fp16",
        *extra,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=REPO_DIR)
    Path(DRIVE_SMOKE, log_name).write_text(proc.stdout + proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(
            f"yolox train exited {proc.returncode}\n{(proc.stdout + proc.stderr)[-2500:]}"
        )


def parse_train_log() -> dict:
    log = Path(RUN_DIR, "train_log.txt").read_text()
    metrics = {}
    losses = re.findall(r"total_loss: ([\d.]+)", log)
    if losses:
        metrics["total_loss_last"] = float(losses[-1])
    ap50_95 = re.findall(
        r"Average Precision\s+\(AP\) @\[ IoU=0\.50:0\.95 \| area=\s*all.*= ([\d.\-]+)", log
    )
    ap50 = re.findall(
        r"Average Precision\s+\(AP\) @\[ IoU=0\.50\s+\| area=\s*all.*= ([\d.\-]+)", log
    )
    if ap50_95:
        metrics["ap50_95"] = float(ap50_95[-1])
    if ap50:
        metrics["ap50"] = float(ap50[-1])
    return metrics


with step("train_1_epoch") as s:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    yolox_train(1, ["--ckpt", str(PRETRAINED_PTH)], "train_1_epoch.log")
    s["checkpoints"] = sorted(p.name for p in RUN_DIR.glob("*.pth"))
    RESULT["metrics"]["after_epoch_1"] = parse_train_log()
    print(s, RESULT["metrics"])

In [ ]:
# --- 7. Resume from latest_ckpt.pth for one more epoch ------------------------------------
# Simulates a Colab disconnect. latest_ckpt.pth stores no numpy AP, so it should load on
# torch >= 2.6 even though the trainer calls torch.load without weights_only (issue #11).
with step("resume_from_checkpoint") as s:
    latest = RUN_DIR / "latest_ckpt.pth"
    if not latest.exists():
        raise RuntimeError("latest_ckpt.pth missing (training step failed?)")
    yolox_train(2, ["--resume", "--ckpt", str(latest)], "resume.log")
    log = Path(RUN_DIR, "train_log.txt").read_text()
    s["saw_epoch_2"] = "epoch: 2/2" in log
    RESULT["metrics"]["after_epoch_2"] = parse_train_log()
    if not s["saw_epoch_2"]:
        raise RuntimeError("resume did not train epoch 2")

In [ ]:
# --- 8. Load checkpoints: default torch.load vs weights_only=False ---------------------------
with step("checkpoint_load") as s:
    import torch

    candidates = [RUN_DIR / n for n in ("best_ckpt.pth", "last_epoch_ckpt.pth")]
    ckpt_path = next((p for p in candidates if p.exists()), None)
    if ckpt_path is None:
        raise RuntimeError(f"no eval checkpoint in {RUN_DIR}")
    s["checkpoint"] = ckpt_path.name

    try:
        torch.load(ckpt_path, map_location="cpu")
        s["default_torch_load"] = "ok"
    except Exception as exc:  # expected on torch >= 2.6 (issue #11)
        s["default_torch_load"] = f"fails: {type(exc).__name__}"

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    s["keys"] = sorted(ckpt)
    s["best_ap"] = float(ckpt.get("best_ap", float("nan")))
    RESULT["metrics"]["best_ap_from_ckpt"] = s["best_ap"]
    print(s)

In [ ]:
# --- 9. ONNX export test -------------------------------------------------------------------
# The bundled export script is broken in 0.4.2 (imports yolox.exp), so call
# torch.onnx.export directly on the nn.Module. Checked with onnxruntime (CPU).
with step("onnx_export") as s:
    import numpy as np
    import torch
    from yolox.models import Yolox

    sys.path.insert(0, f"{REPO_DIR}/exps")
    os.environ.update(PPE_SMOKE_DATA_DIR=DATA_DIR, PPE_SMOKE_OUTPUT_DIR=OUTPUT_DIR)
    from smoke_yolox_s import SmokeYoloxS

    # latest_ckpt.pth: from_pretrained uses a plain torch.load, which fails on AP-carrying ckpts
    wrapper = Yolox.from_pretrained(str(RUN_DIR / "latest_ckpt.pth"), config=SmokeYoloxS())
    module = getattr(wrapper, "module", None)
    if not isinstance(module, torch.nn.Module):
        module = next(v for v in vars(wrapper).values() if isinstance(v, torch.nn.Module))
    module = module.cpu().eval()
    onnx_path = Path(DRIVE_SMOKE, "smoke_yolox_s.onnx")
    dummy = torch.zeros(1, 3, 640, 640)
    torch.onnx.export(
        module,
        dummy,
        str(onnx_path),
        input_names=["images"],
        output_names=["output"],
        opset_version=17,
        dynamo=False,
    )
    s["onnx_mb"] = round(onnx_path.stat().st_size / 1e6, 1)

    run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    import onnxruntime as ort

    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    (out,) = sess.run(None, {"images": dummy.numpy()})
    with torch.no_grad():
        ref = module(dummy).numpy()
    s["output_shape"] = list(out.shape)
    s["max_abs_diff_vs_torch"] = float(np.abs(out - ref).max())
    print(s)

In [ ]:
# --- 10. Summary: copy this JSON into reports/smoke_result.json ------------------------------
RESULT["finished"] = dt.datetime.now(dt.UTC).isoformat(timespec="seconds")
RESULT["all_ok"] = all(v["status"] == "ok" for v in RESULT["steps"].values())
save_result()
print(f"Saved to {RESULT_PATH}\n")
print(json.dumps(RESULT, indent=2, ensure_ascii=False))